In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import importlib
import books
import functions1 
import config 

importlib.reload(books)
importlib.reload(functions1)
importlib.reload(config)

z = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(z)


/Users/alexwebb/laptop_coding/risk_matrix


In [2]:
pf = books.IBKR_live
fetch_csv_robust = functions1.fetch_csv_robust
params = config.params
sort_cols = functions1.sort_cols

GBPUSD = fetch_csv_robust('GBPUSD.FOREX', params=params)
GBPUSD = sort_cols(GBPUSD, ohlc=False)
GBPUSD_last = GBPUSD.iloc[-1]
print('GBPUSD last:', GBPUSD_last)
USD_total = 0
GBP_total = 0
for h in pf:
    exp_usd = h.get("USD_exposure")
    exp_gbp = h.get("GBP_exposure")
    if (exp_usd is None and exp_gbp is None) or h.get('risk_fx') is False:
        print('...skipping', h['name'])
        continue
    print(h['name'],'usd', exp_usd, 'gbp', exp_gbp    )

    # print(h.get("name"),'GBP exp:' ,h.get("GBP_exposure"))
    # get gbpchf last price
    ticker   = h.get("ticker")
    px_df = fetch_csv_robust(f'{ticker}', params=params,)
    px = sort_cols(px_df, ohlc=False).iloc[-1]
    # print('last:' ,px)
    position = h.get("position")
    gpx = .01 if h.get('gbx', True) else 1.0
    mkt_val = position * px * gpx
    print('currency:', h.get('ccy'),'mkt val: £',round(mkt_val))
    if exp_gbp:
        gbp_to_hedge = mkt_val * exp_gbp
        print('gbp to hedge: £',round(gbp_to_hedge))
        GBP_total += gbp_to_hedge

    if exp_usd:
        usd_to_hedge = mkt_val * exp_usd
        if h.get('ccy') == 'GBP':
            print('converting to usd')
            usd_to_hedge *= GBPUSD_last
        USD_total += usd_to_hedge
        print('usd to hedge: $',round(usd_to_hedge))

    print('')
print ('total: £',round(GBP_total))
print ('total: $',round(USD_total))


GBPUSD.FOREX - downloading fresh data
GBPUSD last: 1.3338
XMWX usd None gbp 0.13
currency: GBP mkt val: £ 5997
gbp to hedge: £ 780

...skipping EMIM
VUAG usd 1.0 gbp None
currency: GBP mkt val: £ 3652
converting to usd
usd to hedge: $ 4871

ISJP usd 0 gbp 0.0
currency: JPY mkt val: £ 239692

IEMS usd 0 gbp 0.0
currency: GBP mkt val: £ 610

XXSC usd 0 gbp 0.3
currency: GBP mkt val: £ 1158
gbp to hedge: £ 347

REMX usd 0.3 gbp 0
currency: USD mkt val: £ 330
usd to hedge: $ 99

NOVN usd 0 gbp 0.0
currency: CHF mkt val: £ 1992

...skipping CASH_JPY
total: £ 1127
total: $ 4970
